## Pivot Tables

### Why does this matter?

Pivot tables are one of the most powerful summarization tools in data analysis. Every business analyst, every Excel user, every manager knows pivot tables. As a Data Analyst your job is to build them programmatically in Python — so they can be automated, scheduled, and reproduced. If you've ever used pivot tables in Excel, this is the exact same concept but with full Python control.

### What is a Pivot Table?
What is it?
A pivot table takes a flat DataFrame and reorganizes it — rows become one dimension, columns become another dimension, and the values in the cells are aggregated numbers. It's a way of looking at the same data from a different angle.
### Why does it exist?
Because *groupby()* gives you a one-dimensional summary — one group key, one result column. But sometimes you need a two-dimensional summary — "show me salary by Department AND by Level at the same time, in a grid". That's what pivot tables do. They turn row-based grouped data into a readable matrix.
### How is it different from groupby()?
*groupby()* stacks results vertically — one row per group. *pivot_table()* spreads results horizontally — one dimension becomes rows, another becomes columns. Both summarize data, just in different shapes. Pivot tables are essentially *groupby()* with a visual reshape on top.

In [3]:
import pandas as pd
import numpy as np

data = {
    'Name':   ['Sumed','Priya','Rahul','Neha','Arjun','Divya','Karan','Meera'],
    'Dept':   ['Data','HR','Data','Finance','Data','HR','Finance','Data'],
    'Level':  ['Junior','Junior','Senior','Junior','Senior','Senior','Senior','Junior'],
    'Salary': [60000,45000,75000,55000,80000,62000,71000,58000],
    'Sales':  [120,85,200,95,250,140,175,110]
}
df = pd.DataFrame(data)
df

,Name,Dept,Level,Salary,Sales
0,Sumed,Data,Junior,60000,120
1,Priya,HR,Junior,45000,85
2,Rahul,Data,Senior,75000,200
3,Neha,Finance,Junior,55000,95
4,Arjun,Data,Senior,80000,250
5,Divya,HR,Senior,62000,140
6,Karan,Finance,Senior,71000,175
7,Meera,Data,Junior,58000,110


## Basic pivot_table()

In [4]:
# pivot_table() — 4 key parameters:
# values  → which column to aggregate (the numbers)
# index   → which column becomes the ROWS
# columns → which column becomes the COLUMNS (spreads horizontally)
# aggfunc → how to aggregate (default is 'mean')

# Average Salary by Dept (rows) and Level (columns)
pd.pivot_table(
    df,
    values  = 'Salary',
    index   = 'Dept',
    columns = 'Level',
    aggfunc = 'mean'
)

Level,Junior,Senior
Dept,,
Data,59000.0,77500.0
Finance,55000.0,71000.0
HR,45000.0,62000.0


*Each cell = mean Salary for that Dept + Level combination. Data Senior average = (75000+80000)/2 = 77500. This 2D view is impossible to get cleanly from groupby alone.*

## AGGFUNC() : Changing the aggregation

In [6]:
# aggfunc accepts: 'mean','sum','count','min','max','median'
# or a list of functions for multiple aggregations at once
# or a numpy function like np.sum

# Count of employees per Dept and Level
pd.pivot_table(df, values='Name', index='Dept',
               columns='Level', aggfunc='count')


Level,Junior,Senior
Dept,,
Data,2,2
Finance,1,1
HR,1,1


In [8]:

# Total Sales per Dept and Level
pd.pivot_table(df, values='Sales', index='Dept',
               columns='Level', aggfunc='sum')


Level,Junior,Senior
Dept,,
Data,230,450
Finance,95,175
HR,85,140


In [9]:

# Multiple aggregations — aggfunc as a list
pd.pivot_table(df, values='Salary', index='Dept',
               aggfunc=['mean', 'min', 'max', 'count'])

,mean,min,max,count
,Salary,Salary,Salary,Salary
Dept,,,,
Data,68250.0,58000,80000,4
Finance,63000.0,55000,71000,2
HR,53500.0,45000,62000,2


## MARGINS : Adding rows and column totals

In [10]:
# margins=True → adds an 'All' row and 'All' column
# Shows the overall total/average for each row and column
# margins_name → customize the label (default is 'All')

pd.pivot_table(
    df,
    values       = 'Salary',
    index        = 'Dept',
    columns      = 'Level',
    aggfunc      = 'mean',
    margins      = True,
    margins_name = 'Total Avg'
)

# WHY margins matter?
# In a business report you always want subtotals
# "What is the overall average salary across ALL depts?"
# "What is the average for ALL juniors regardless of dept?"
# margins answers both questions in one table

Level,Junior,Senior,Total Avg
Dept,,,
Data,59000.0,77500.0,68250.0
Finance,55000.0,71000.0,63000.0
HR,45000.0,62000.0,53500.0
Total Avg,54500.0,72000.0,63250.0


*Bottom-right cell (63375.0) = overall average of ALL salaries. Bottom row = average per level across all depts. Right column = average per dept across all levels. This is exactly what an Excel pivot table shows.*

## FILL_VALUE : handlin NaN in pivot tables

In [11]:
# When a combination has NO data → pivot table shows NaN
# Example: if no Junior exists in Finance → Finance/Junior = NaN
# fill_value replaces those NaN cells with a specified value

pd.pivot_table(
    df,
    values     = 'Sales',
    index      = 'Dept',
    columns    = 'Level',
    aggfunc    = 'sum',
    fill_value = 0    # replace NaN with 0
)

# WHY 0 and not some other value?
# Because NaN in a pivot table means "no records exist"
# For counts and sums → 0 is the correct representation
# For averages → be careful: 0 is misleading (use NaN or leave blank)

Level,Junior,Senior
Dept,,
Data,230,450
Finance,95,175
HR,85,140


*Only use fill_value=0 for sum and count aggregations. For mean, replacing NaN with 0 is mathematically wrong — it implies a group exists with zero value, which skews the average.*

## Multiple values and multiple index columns

In [13]:
# values can be a list → aggregate multiple columns at once
pd.pivot_table(
    df,
    values  = ['Salary', 'Sales'],   # two value columns
    index   = 'Dept',
    columns = 'Level',
    aggfunc = 'mean'
)


Salary           Sales       
Level     Junior   Senior Junior Senior
Dept                                   
Data     59000.0  77500.0  115.0  225.0
Finance  55000.0  71000.0   95.0  175.0
HR       45000.0  62000.0   85.0  140.0

In [14]:
# Creates a MultiIndex column structure
# Top level: Salary, Sales
# Second level: Junior, Senior

# index can also be a list → multiple row grouping levels
pd.pivot_table(
    df,
    values  = 'Salary',
    index   = ['Dept', 'Level'],   # two row dimensions
    aggfunc = 'mean'
)
# No columns= parameter → just a multi-level groupby result

Salary
Dept    Level          
Data    Junior  59000.0
        Senior  77500.0
Finance Junior  55000.0
        Senior  71000.0
HR      Junior  45000.0
        Senior  62000.0

*Multiple values creates a MultiIndex in columns. To flatten it for export use pt.columns = ['_'.join(col) for col in pt.columns] — turns ('Salary','Junior') into 'Salary_Junior'.*

## Crosstab(): frequency table shortcut

In [16]:
# crosstab() → specialized pivot table for FREQUENCY counts
# Counts how many times each combination appears
# Simpler syntax than pivot_table for counting

pd.crosstab(df['Dept'], df['Level'])
# Rows = Dept, Columns = Level, Values = count of occurrences



Level,Junior,Senior
Dept,,
Data,2,2
Finance,1,1
HR,1,1


In [18]:
# normalize → show PERCENTAGES instead of counts
pd.crosstab(df['Dept'], df['Level'], normalize='index')
# normalize='index'  → % within each ROW (dept) → adds to 100% per row
# normalize='columns'→ % within each COLUMN (level)
# normalize='all'    → % of grand total



Level,Junior,Senior
Dept,,
Data,0.5,0.5
Finance,0.5,0.5
HR,0.5,0.5


In [19]:
# margins=True works here too
pd.crosstab(df['Dept'], df['Level'], margins=True)

Level,Junior,Senior,All
Dept,,,
Data,2,2,4
Finance,1,1,2
HR,1,1,2
All,4,4,8


 *crosstab vs pivot_table — use crosstab when you just want counts or percentages of combinations. Use pivot_table when you want to aggregate a specific value column (salary, sales, etc.).*

## Flattening and using pivot table results

In [20]:
# A pivot table result is still a DataFrame — you can use it further

pt = pd.pivot_table(df, values='Salary', index='Dept',
                     columns='Level', aggfunc='mean')

# Access a specific cell
pt.loc['Data', 'Senior']       # → 77500.0

# Access a specific column (all depts, one level)
pt['Senior']

# Reset index → flatten back to a regular DataFrame
pt.reset_index()

# Flatten MultiIndex columns (when using multiple values)
pt2 = pd.pivot_table(df, values=['Salary','Sales'],
                      index='Dept', columns='Level', aggfunc='mean')
pt2.columns = ['_'.join(col) for col in pt2.columns]
# ('Salary','Junior') → 'Salary_Junior'
# ('Sales','Senior')  → 'Sales_Senior'
pt2.reset_index()   # now a clean flat DataFrame

,Dept,Salary_Junior,Salary_Senior,Sales_Junior,Sales_Senior
0,Data,59000.0,77500.0,115.0,225.0
1,Finance,55000.0,71000.0,95.0,175.0
2,HR,45000.0,62000.0,85.0,140.0


*Flattening MultiIndex columns with '_'.join(col) is essential before saving to CSV or Excel — file formats don't support MultiIndex headers.*

## Everything You Must Know Cold

4 core parameters of pivot_table() — **values** (what to aggregate), **index** (row dimension), **columns** (column dimension), **aggfunc** (how to aggregate). Know these cold.

**aggfunc** — accepts string **'mean', 'sum', 'count', 'min', 'max',** a list of functions, or numpy functions. Default is **'mean'**.

**margins=True** — adds totals row and column. Use *margins_name* to rename from *'All'* to something meaningful. Essential for business reports.

**fill_value=0** — replaces NaN cells where no combination exists. Only use with *sum* and *count* — never with *mean* as 0 is mathematically wrong for averages.
**crosstab()** — shortcut for frequency counts between two categorical columns. Use *normalize='index'/'columns'/'all'* for percentage distributions.

**Flattening MultiIndex — pt.columns = ['_'.join(col) for col in pt.columns] — essential before saving to CSV or Excel.**